#### [Notebook 2 Quick Start](https://confluence.shopee.io/x/7O0lhw)
#### Common Use Cases and Solution Using Notebook 2
- [Multi Kernels in Notebook 2](https://confluence.shopee.io/x/4Od2hw)
- [Execute HDFS command](https://confluence.shopee.io/x/P5w8hw)
- [Browse, upload and download from Data Infra S3](https://confluence.shopee.io/x/xo9Ohw)
- [Setting up 3rd party packages for your Notebook 2](https://confluence.shopee.io/x/JZFOhw)
- [Develop and setup user defined function or library](https://confluence.shopee.io/x/hpjehg)
- [Execute Presto/Trino query directly in Python Kernel Notebook](https://confluence.shopee.io/x/WpMlhw)

> Note: "Python on K8S" Kernel cannot execute Spark functions. ***If you want to use Spark, change kernel to Spark***. [How to change Kernel](https://confluence.shopee.io/x/4Od2hw#MultiKernelsinNotebook2-SetKernelwhencreatingnewNotebook2file)

In [ ]:
import pandas as pd
import re,json,sys,os
from io import StringIO
import argparse
import sys
import os
import re
import datetime 
from datetime import datetime, timedelta
import json
import requests
import pytz
from collections import defaultdict
from IPython import get_ipython
from IPython.utils.io import capture_output
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, input_file_name, lit
import json
import logging
import requests

In [ ]:
# 获取df
spark = SparkSession.builder \
            .appName('ods_upstream_monitor') \
            .master('yarn') \
            .config('spark.sql.execution.arrow.pyspark.enabled', True) \
            .config('spark.sql.session.timeZone', 'UTC') \
            .config('spark.driver.memory','8G') \
            .config('spark.executor.memory', '8g') \
            .config('spark.executor.cores', '2') \
            .config('spark.num.executors', '32') \
            .config('spark.ui.showConsoleProgress', True) \
            .config('spark.sql.repl.eagerEval.enabled', True) \
            .enableHiveSupport() \
            .getOrCreate()

sql = """
        select *
        from risk_feature.dim_risk_user_biz_manage_reg_tab
        where cluster = 'SG' and mart_schema = 'risk_feature'
        ;
       """
dqc_df = spark.sql(sql)

In [ ]:
def seatalk_alert(alert_msg, email_list=[], seatalk_url="https://openapi.seatalk.io/webhook/group/8MAvH5UFSR2V5pCoDO5BZQ"):
    content = str(alert_msg)
    post_data = {
        "tag": "text",
        "text": {
            "content": content,
        }
    }
    if len(email_list) != 0:
        post_data['text']['mentioned_email_list'] = email_list
    data = json.dumps(post_data)
    headers = {'Content-Type': 'application/json'}
    response = requests.request("POST", url=seatalk_url, headers=headers, data=data)
    logging.info(f"seatalk_alert response: {response.text}")
    result_json = json.loads(response.text)

    if result_json["code"] == 3:
        seatalk_alert(content[:1000] + "...more...", seatalk_url, email_list)


In [ ]:
def alert_from_spark_df(spark_df):
    """
    遍历 Spark DataFrame 的每一行并调用 seatalk_alert
    """
    # ds = (datetime.today() - timedelta(days=1)).strftime('%Y-%m-%d')
    # ds_t_2 = (datetime.today() - timedelta(days=2)).strftime('%Y-%m-%d')
    
    rows = spark_df.collect()  # ⚠️ 收集到 driver，适用于数据量小的情况
    for row in rows:
        pic = [row['pic']]
        remark = f"集群：{row['cluster']} | Schema：{row['biz_schema']}\n Table：{row['biz_table']}"
        sql = """
SELECT 
    mart_table,
    MAX(IF(grass_date = '${ds}', cnt, 0)) AS t_1_cnt,
    MAX(IF(grass_date = '${ds_t_2}', cnt, 0)) AS t_2_cnt,
    '${ds}' as grass_date,
    -- 新增百分比（数字类型）
    IF(
        (MAX(IF(grass_date = '${ds}', cnt, 0)) - MAX(IF(grass_date = '${ds_t_2}', cnt, 0)))
        / MAX(IF(grass_date = '${ds_t_2}', cnt, 0))
        > ABS(MAX(CAST(volume_increasement_criteria_ AS DOUBLE))), 1, 0
    ) AS t_increase_out_bound_tag,
    
    -- 减少百分比（数字类型）
    IF(
        (MAX(IF(grass_date = '${ds}', cnt, 0)) - MAX(IF(grass_date = '${ds_t_2}', cnt, 0)))
        / MAX(IF(grass_date = '${ds_t_2}', cnt, 0))
        < -ABS(MAX(CAST(volume_decreasement_criteria_ AS DOUBLE))), 1, 0
    ) AS t_decrease_out_bound_tag,
    
    -- 新增百分比
    CONCAT(
        ROUND(
            CASE 
                WHEN MAX(IF(grass_date = '${ds}', cnt, 0)) > MAX(IF(grass_date = '${ds_t_2}', cnt, 0)) 
                THEN (MAX(IF(grass_date = '${ds}', cnt, 0)) - MAX(IF(grass_date = '${ds_t_2}', cnt, 0)))
                     / MAX(IF(grass_date = '${ds_t_2}', cnt, 0)) * 100
                ELSE 0
            END, 4
        ), 
        '%'
    ) AS t_increase_pct,
    
    -- 减少百分比
    CONCAT(
        ROUND(
            CASE 
                WHEN MAX(IF(grass_date = '${ds}', cnt, 0)) < MAX(IF(grass_date = '${ds_t_2}', cnt, 0)) 
                THEN (MAX(IF(grass_date = '${ds_t_2}', cnt, 0)) - MAX(IF(grass_date = '${ds}', cnt, 0)))
                     / MAX(IF(grass_date = '${ds_t_2}', cnt, 0)) * 100
                ELSE 0
            END, 4
        ), 
        '%'
    ) AS t_decrease_pct

FROM (
    SELECT 
        t1.*, 
        t2.volume_decreasement_criteria_, 
        t2.volume_increasement_criteria_
    FROM
    (  
        SELECT 
            COUNT(1) AS cnt, 
            grass_date,
            '{mart_table}' AS mart_table, 
            '{biz_table}' AS biz_table, 
            '{biz_schema}' AS biz_schema
        FROM {mart_schema}.{mart_table} 
        WHERE grass_date >= '${ds_t_2}' 
        GROUP BY grass_date
    ) t1 
    LEFT JOIN 
    (
        SELECT
            biz_schema,
            biz_table,
            CASE 
                WHEN volume_decreasement_criteria__ LIKE '%\%%' THEN CAST(REGEXP_REPLACE(volume_decreasement_criteria__, '%', '') AS DOUBLE) / 100
                ELSE CAST(volume_decreasement_criteria__ AS DOUBLE)
            END AS volume_decreasement_criteria_,
            
            CASE 
                WHEN volume_increasement_criteria__ LIKE '%\%%' THEN CAST(REGEXP_REPLACE(volume_increasement_criteria__, '%', '') AS DOUBLE) / 100
                ELSE CAST(volume_increasement_criteria__ AS DOUBLE)
            END AS volume_increasement_criteria_
        FROM {mart_schema}.dim_risk_user_biz_manage_reg_alarm_criteria_df 
        WHERE biz_table = '{biz_table}'
    ) t2 
    ON t1.biz_table = t2.biz_table AND t1.biz_schema = t2.biz_schema 
) tmp 
GROUP BY mart_table

           """.format(mart_table=row['mart_table'],mart_schema = row['mart_schema'], biz_table= row['biz_table'], biz_schema = row['biz_schema'])
        dqc_df = spark.sql(sql)
        dqc_df = dqc_df[(dqc_df['t_increase_out_bound_tag'] == 1) | (dqc_df['t_decrease_out_bound_tag'] == 1)]
        print(dqc_df)
        for row in dqc_df.collect():

            # 判断减少/新增百分比
            decrease_pct = float(row['t_decrease_pct'].replace('%', ''))/100
            increase_pct = float(row['t_increase_pct'].replace('%', ''))/100

            alert_msg = (
                f"【Risk业务透传表同步告警】\n"
                f"业务数据切片日期: {row['grass_date']}\n"
                f"{remark}\n"
                f"业务表数量：{row['t_1_cnt']}\n"
                f"数仓表数量：{row['t_2_cnt']}\n"
            )
            
            if decrease_pct > 0:
                display_increase = "None"   			# 显示新增百分比
                display_decrease = f"{decrease_pct*100:.2f}%"    # 显示减少百分比
                alert_msg += f"减少百分比：{display_decrease}\n"
            elif increase_pct > 0:
                display_decrease = "None"   			# 显示新增百分比
                display_increase = f"{increase_pct*100:.2f}%"    # 显示减少百分比
                alert_msg += f"新增百分比：{display_increase}\n"
            else:
                display_decrease = "None"               # 减少百分比置为 None
                display_increase = "None"               # 新增百分比置为 None

            seatalk_alert(alert_msg, email_list=pic)
            print(alert_msg)
            print(row['mart_table'])
            print(" this tab has violate the rule")
            print(sql)
            
               
    print("✅ 所有行的 Seatalk 告警已发送完毕。")

In [ ]:
alert_from_spark_df(dqc_df)